# FASTT — main results on AMI-SDM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/main_ami_sdm.ipynb)

This notebook reproduces the paper's main experiment on AMI-SDM. Every method's hyperparameters are selected on
a 25 % hold-out of the corpus; every method is then trained and tested on the same ten folds of a 5×2
cross-validation of the remaining clips, and the two FASTT variants are compared with every other method on those
folds.

The configuration is fixed in `fastt.configs.MAIN["ami_sdm"]` and the code is in `fastt.experiment`. The
features and the stored searches and folds are read from the Hugging Face dataset
[`huseyin-karaca/fastt`](https://huggingface.co/datasets/huseyin-karaca/fastt); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold refitted from the published features (`RETUNE = True`: the search as well) | a CPU, hours |
| **3** | level 2 on features rebuilt from the audio: run `extract` first on the same machine, which writes them to a local folder that level 3 reads (`FASTT_HUB` names another place) | a GPU for `extract` |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [2]:
from fastt.configs import MAIN
from fastt.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again

exp = MainExperiment(MAIN["ami_sdm"], level=LEVEL, retune=RETUNE)

AMI-SDM: level 1 — search read from the Hub, folds read from the Hub
experts  ESPnet Conformer-RNN-T, ESPnet Transformer, wav2vec2-large-960h
arms     FASTT-Alternating, FASTT-Boosted, Raw XGBoost (AdaSTT), Raw SDT, SelectKBest XGBoost, SelectKBest SDT, ROVER (weighted), ROVER (confidence-weighted), CN-MBR, Score-Fusion, Hard-Selector (plain CE), Hard-Selector (cost-sensitive CE), Soft-Selector (mixture), QE-Router


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every method and their choices; every other parameter is fixed
(`fastt.configs.SPACES`). The spaces are the same on every corpus.

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed
  [FASTT-Alternating  (18 free of 18)]
           parameter kind                                   choices / range
      transform_type  cat {nonlinear, linear, diagonal, low_rank, identity}
      num_iterations  cat                                     {6, 12, 3, 1}
        transform_lr  cat                             {0.001, 0.0001, 0.01}
     transform_steps  cat                                    {100, 25, 400}
        surrogate_lr  cat                                {0.01, 0.1, 0.001}
     surrogate_steps  cat                                    {50, 200, 400}
             lambda1  cat                               {0.1, 0.001, 1e-05}
             lambda2  cat                               {1e-05, 0.1, 0.001}
    sel_n_estimators  cat                                             {400}
   sel_learning_rate  cat                            {0.03, 0.3, 0.1, 0.01}
       sel_max_depth  cat                                     {5, 12, 8, 3}
       sel_subsample  cat                                   {1.0, 0.6, 0.8}
       sel_colsample  cat                              {0.6, 1.0, 0.3, 0.1}
sel_min_child_weight  cat                                             {1.0}
      sel_reg_lambda  cat                                             {3.0}
        weight_decay  cat                                            {0.01}
           out_ratio  cat                                   {1.0, 0.1, 0.3}
    bottleneck_ratio  cat                                            {0.05}
  [FASTT-Boosted  (15 free of 15)]
        parameter kind                                   choices / range
   transform_type  cat {identity, low_rank, linear, nonlinear, diagonal}
          lambda1  cat                                           {0.001}
          lambda2  cat                               {0.001, 1e-05, 0.1}
        out_ratio  cat                                             {0.1}
 bottleneck_ratio  cat                                 {0.15, 0.5, 0.05}
       num_rounds  cat                                      {6, 1, 4, 2}
        num_trees  cat                                    {24, 8, 16, 4}
            depth  cat                                               {7}
            lmbda  cat                        {0.01, 0.0001, 1.0, 1e-06}
           lmbda2  cat                                           {1e-07}
          adam_lr  cat                                          {0.0003}
adam_weight_decay  cat                        {0.001, 1e-07, 1e-05, 0.1}
       batch_size  cat                                  {512, 128, 2048}
         patience  cat                                      {40, 20, 10}
   t_weight_decay  cat                                           {1e-06}
  [Raw XGBoost (AdaSTT)  (7 free of 7)]
           parameter kind        choices / range
    sel_n_estimators  cat   {1000, 400, 150, 50}
   sel_learning_rate  cat {0.3, 0.1, 0.03, 0.01}
       sel_max_depth  cat                    {3}
       sel_subsample  cat        {1.0, 0.8, 0.6}
       sel_colsample  cat                  {1.0}
sel_min_child_weight  cat                 {16.0}
      sel_reg_lambda  cat       {3.0, 0.3, 30.0}
  [Raw SDT  (14 free of 15)]
        parameter kind              choices / range
   t_weight_decay  cat                      {1e-06}
          lambda1  cat                      {0.001}
          lambda2  cat                      {0.001}
        out_ratio  cat                        {0.1}
 bottleneck_ratio  cat                       {0.05}
       num_rounds  cat                 {1, 2, 4, 6}
        num_trees  cat                          {8}
            depth  cat                 {2, 7, 5, 3}
            lmbda  cat   {1e-06, 1.0, 0.0001, 0.01}
           lmbda2  cat                        {0.1}
          adam_lr  cat {0.001, 0.003, 0.0003, 0.01}
adam_weight_decay  cat                      {1e-07}
       batch_size  cat             {2048, 512, 128}
         patience  cat              

## 2. Data

The three experts' pooled encoder features, their transcripts, and the transcripts' word-error counters against
the reference, for every clip. A sample of the stored counters is recomputed from the stored transcripts as a
check.

In [4]:
exp.load()

41,641 utterances x 3 routable model(s), from cache .fastt-cache/store_f06450c105e3398c
texts for 3 model(s), from cache
100.0% of 1,200 (utterance, model) counters reproduce exactly


EnsembleStore(ami_sdm, 'mean', 41,641 utterances, 3 routable)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every method, ten configurations are drawn at random
from its space (Table 2), each fitted on 70 % of the hold-out and scored on the remaining 30 %; the configuration
with the lowest corpus WER is kept. When several tie on it, the one the paper's run kept is used
(`fastt.configs.SELECTED`), after checking that it is among the tied best. The other 75 % of the clips are never
seen by the search.

In [5]:
exp.search()


=== best fastt_alternating   corpus WER 0.6189   gap closed +28.5%   (10 trials, 0.0 min)
{
  "lambda1": 0.1,
  "lambda2": 0.001,
  "num_iterations": 3,
  "sel_colsample": 0.6,
  "sel_learning_rate": 0.1,
  "sel_max_depth": 5,
  "sel_min_child_weight": 1.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 3.0,
  "sel_subsample": 0.6,
  "surrogate_lr": 0.01,
  "surrogate_steps": 400,
  "transform_lr": 0.01,
  "transform_steps": 400,
  "transform_type": "diagonal"
}

=== best fastt_boosted   corpus WER 0.6182   gap closed +29.2%   (10 trials, 0.0 min)
{
  "adam_lr": 0.0003,
  "adam_weight_decay": 0.001,
  "batch_size": 512,
  "depth": 7,
  "epochs": 200,
  "lmbda": 1.0,
  "lmbda2": 1e-07,
  "num_rounds": 2,
  "num_trees": 8,
  "patience": 20,
  "transform_type": "identity"
}

=== best adastt   corpus WER 0.6206   gap closed +26.5%   (10 trials, 0.0 min)
{
  "sel_colsample": 1.0,
  "sel_learning_rate": 0.1,
  "sel_max_depth": 3,
  "sel_min_child_weight": 16.0,
  "sel_n_estimators": 400,
  


=== best rover_w   corpus WER 0.6648   gap closed -22.8%   (10 trials, 0.0 min)
{
  "alpha": 1.0,
  "conf_source": "cn",
  "null_penalty": 0.5,
  "weight_by_conf": false,
  "weighting": "inverse_wer"
}

=== best rover_conf   corpus WER 0.6635   gap closed -21.4%   (10 trials, 0.0 min)
{
  "alpha": 0.25,
  "conf_source": "cn",
  "conf_temp": 0.25,
  "null_penalty": 0.25,
  "weight_by_conf": false,
  "weighting": "inverse_wer"
}

=== best cn_mbr   corpus WER 0.6597   gap closed -17.2%   (10 trials, 0.0 min)
{
  "candidates": "members",
  "conf_source": "blend",
  "decode": "hyp",
  "null_prior": 0.25,
  "prior": "inverse_wer",
  "temperature": 10.0,
  "use_conf": true
}

=== best score_fusion   corpus WER 0.6754   gap closed -34.7%   (10 trials, 0.0 min)
{
  "mode": "hard",
  "w_conf": 1.0,
  "w_consensus": -1.0,
  "w_length": -1.0
}

=== best argmin_clf   corpus WER 0.6227   gap closed +24.1%   (10 trials, 0.0 min)
{
  "alpha": 0.1
}

=== best probe   corpus WER 0.6302   gap closed +15

partition,n_utts,share
tune,10410,0.250
eval,31231,0.750
fit,7287,0.175
val,3123,0.075


  on the tuning-score rows: bsm(espnet_transducer) 0.6443   oracle 0.5548   gap 0.0895
  the 31,231 evaluation rows are never seen by a trial
  objectives   : corpus_wer (minimize) — crossval.METRICS['corpus_wer'], on the tuning-score rows
  14 arms x 10 trials, read from the Hub; 31,231 evaluation clips the search never saw


HPTRun('hpt_ami_sdm_cd75c8ac', 6 arm(s), best fastt_boosted 0.6182, eval on 31,231 rows)

In [6]:
exp.dataset()

statistic,ami_sdm
Total clips,41641
Features,2049
Hyperparameter hold-out,10410
Evaluation rows (5x2),31231
Train size (per fold),15615
Test size (per fold),15616
Hours (est.),14.4890
Reference words,315864


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
FASTT-Alternating,10,0.6189,"{""lambda1"": 0.1, ""lambda2"": 0.001, ""num_iterations"": 3, ""sel_colsample"": 0.6, ""sel_learning_rate"": 0.1, ""sel_max_depth"": 5, ""sel_min_child_weight"": 1.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 3.0, ""sel_subsample"": 0.6, ""surrogate_lr"": 0.01, ""surrogate_steps"": 400, ""transform_lr"": 0.01, ""transform_steps"": 400, ""transform_type"": ""diagonal""}"
FASTT-Boosted,10,0.6182,"{""adam_lr"": 0.0003, ""adam_weight_decay"": 0.001, ""batch_size"": 512, ""depth"": 7, ""epochs"": 200, ""lmbda"": 1.0, ""lmbda2"": 1e-07, ""num_rounds"": 2, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
Raw XGBoost (AdaSTT),10,0.6206,"{""sel_colsample"": 1.0, ""sel_learning_rate"": 0.1, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 0.3, ""sel_subsample"": 0.6}"
Raw SDT,10,0.6443,"{""adam_lr"": 0.001, ""adam_weight_decay"": 1e-07, ""batch_size"": 512, ""depth"": 5, ""epochs"": 200, ""lmbda"": 0.0001, ""lmbda2"": 0.1, ""num_rounds"": 2, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
SelectKBest XGBoost,10,0.6275,"{""k"": 128, ""sel_colsample"": 1.0, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 1000, ""sel_reg_lambda"": 30.0, ""sel_subsample"": 0.8}"
SelectKBest SDT,10,0.6443,"{""adam_lr"": 0.003, ""adam_weight_decay"": 0.1, ""batch_size"": 2048, ""depth"": 7, ""epochs"": 200, ""k"": 32, ""lmbda"": 1e-06, ""lmbda2"": 0.1, ""num_rounds"": 4, ""num_trees"": 24, ""patience"": 20, ""transform_type"": ""identity""}"
ROVER (weighted),10,0.6648,"{""alpha"": 1.0, ""conf_source"": ""cn"", ""null_penalty"": 0.5, ""weight_by_conf"": false, ""weighting"": ""inverse_wer""}"
ROVER (confidence-weighted),10,0.6635,"{""alpha"": 0.25, ""conf_source"": ""cn"", ""conf_temp"": 0.25, ""null_penalty"": 0.25, ""weight_by_conf"": false, ""weighting"": ""inverse_wer""}"
CN-MBR,10,0.6597,"{""candidates"": ""members"", ""conf_source"": ""blend"", ""decode"": ""hyp"", ""null_prior"": 0.25, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": true}"
Score-Fusion,10,0.6754,"{""mode"": ""hard"", ""w_conf"": 1.0, ""w_consensus"": -1.0, ""w_length"": -1.0}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every method is trained
on one half and tested on the other, then the other way round: ten folds, the same for every method.

In [8]:
exp.cross_validate();

  runlog HIT  cv5x2 seed 11 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 22 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 33 — 15/15 group(s) cached, nothing refitted


  runlog HIT  cv5x2 seed 44 — 15/15 group(s) cached, nothing refitted


  runlog HIT  cv5x2 seed 55 — 15/15 group(s) cached, nothing refitted
5 repetitions x 2 folds x 24 arms, read from the Hub


## 5. Main results (Table 3)

Corpus WER and mean utterance WER (mean ± sd over the ten folds), the share of the oracle gap closed and the
selection accuracy at 0, 1 and 2 words of tolerance, and how often each method picks each expert.

In [9]:
exp.results()

Table 3 — main results, AMI-SDM
mean ± sd over 10 folds; GC / uGC / Acc in %; dist = share of clips routed to ESPnet Conformer-RNN-T / ESPnet Transformer / wav2vec2-large-960h; best non-oracle value per column in bold
  [individual]
                                       arm             WER            uWER        GC@0        GC@1        GC@2        uGC@0        uGC@1         uGC@2      Acc@0      Acc@1      Acc@2           dist
                    ESPnet Conformer-RNN-T 0.6528 ± 0.0033 0.8889 ± 0.0025   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0    0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0 66.4 ± 0.4 82.0 ± 0.3 90.7 ± 0.1 1.00-0.00-0.00
                        ESPnet Transformer 0.7002 ± 0.0021 0.9040 ± 0.0034 -50.3 ± 2.5 -64.3 ± 3.3 -93.0 ± 4.7  -11.6 ± 2.6  -18.5 ± 4.3   -33.8 ± 7.6 57.7 ± 0.3 74.1 ± 0.2 83.6 ± 0.2 0.00-1.00-0.00
                       wav2vec2-large-960h 0.6594 ± 0.0026 1.0344 ± 0.0029  -7.1 ± 1.2  -9.0 ± 1.5 -13.0 ± 2.2 -111.2 ± 3.6 -177.7 ± 7.1 -324.8 ± 19.5 59.8 ± 0.4 81.4 ± 0.1 91.1 ± 0.1 0.00-0.00-1.00
Best-Single-Model (ESPnet Conformer-RNN-T) 0.6528 ± 0.0033 0.8889 ± 0.0025   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0    0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0 66.4 ± 0.4 82.0 ± 0.3 90.7 ± 0.1 1.00-0.00-0.00
  [external]
                              arm             WER            uWER        GC@0        GC@1         GC@2        uGC@0        uGC@1         uGC@2      Acc@0      Acc@1      Acc@2           dist
                  Random-Ensemble 0.6710 ± 0.0031 0.9432 ± 0.0014 -19.3 ± 1.5 -24.7 ± 2.0  -35.7 ± 3.0  -41.6 ± 2.7  -66.5 ± 4.5  -121.4 ± 9.7 61.2 ± 0.3 79.2 ± 0.1 88.5 ± 0.2 0.33-0.33-0.33
                 ROVER (weighted) 0.6667 ± 0.0092 1.0602 ± 0.0042 -14.7 ± 6.6 -18.8 ± 8.4 -27.2 ± 12.1 -130.9 ± 3.3 -209.3 ± 7.1 -382.4 ± 21.6          —          —          —              —
      ROVER (confidence-weighted) 0.6683 ± 0.0034 1.0150 ± 0.0022 -16.4 ± 1.2 -21.0 ± 1.5  -30.3 ± 2.3  -96.4 ± 2.7 -154.0 ± 5.3 -281.3 ± 12.5          —          —          —              —
                           CN-MBR 0.6680 ± 0.0031 1.0625 ± 0.0031 -16.2 ± 0.9 -20.7 ± 1.2  -29.9 ± 1.7 -132.7 ± 3.0 -212.1 ± 6.4 -387.5 ± 19.7 57.4 ± 0.3 79.5 ± 0.1 90.1 ± 0.1 0.32-0.11-0.57
                     Score-Fusion 0.6796 ± 0.0035 0.9409 ± 0.0031 -28.4 ± 1.6 -36.4 ± 2.2  -52.6 ± 3.2  -39.7 ± 2.5  -63.5 ± 4.3  -116.1 ± 9.9 60.0 ± 0.3 77.7 ± 0.2 87.1 ± 0.2 0.28-0.32-0.40
         Hard-Selector (plain CE) 0.6284 ± 0.0038 0.8771 ± 0.0026  25.9 ± 1.0  33.1 ± 1.3   47.8 ± 1.8    9.0 ± 0.7   14.4 ± 1.0    26.2 ± 1.9 71.3 ± 0.4 86.5 ± 0.3 94.0 ± 0.2 0.84-0.02-0.14
Hard-Selector (cost-sensitive CE) 0.6392 ± 0.0052 0.8830 ± 0.0032  14.4 ± 2.7  18.4 ± 3.4   26.6 ± 5.0    4.5 ± 1.1    7.2 ± 1.8    13.1 ± 3.4 69.3 ± 0.8 84.6 ± 0.6 92.4 ± 0.4 0.87-0.00-0.13
                        QE-Router 0.6272 ± 0.0025 0.9006 ± 0.0031  27.1 ± 0.9  34.6 ± 1.1   50.0 ± 1.5   -8.9 ± 1.6  -14.3 ± 2.4   -26.1 ± 4.9 70.6 ± 0.3 87.2 ± 0.1 94.4 ± 0.1 0.33-0.24-0.43
  [classical]
                 arm             WER            uWER       GC@0       GC@1       GC@2      uGC@0      uGC@1      uGC@2      Acc@0      Acc@1      Acc@2           dist
Raw XGBoost (AdaSTT) 0.6274 ± 0.0029 0.8585 ± 0.0024 26.9 ± 1.6 34.4 ± 2.0 49.7 ± 2.7 23.2 ± 1.1 37.1 ± 1.9 67.7 ± 3.7 72.2 ± 0.2 86.9 ± 0.2 93.9 ± 0.1 0.47-0.28-0.26
             Raw SDT 0.6528 ± 0.0033 0.8889 ± 0.0025  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0 66.4 ± 0.4 82.0 ± 0.3 90.7 ± 0.1 1.00-0.00-0.00
 SelectKBest XGBoost 0.6343 ± 0.0033 0.8663 ± 0.0020 19.5 ± 1.8 24.9 ± 2.2 36.0 ± 3.1 17.3 ± 1.4 27.6 ± 2.3 50.4 ± 4.2 70.4 ± 0.4 85.5 ± 0.2 93.0 ± 0.2 0.49-0.30-0.21
     SelectKBest SDT 0.6528 ± 0.0033 0.8889 ± 0.0025  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0 66.4 ± 0.4 82.0 ± 0.3 90.7 ± 0.1 1.00-0.00-0.00
  [fastt]
              arm             WER            uWER       GC@0       GC@1       GC@2      uGC@0      uGC@1      uGC@2      Acc@0      Acc@1      Acc@2           dist
    FASTT-Boosted

In [10]:
exp.routing()

model,n_picks,n_fused,share_espnet_transducer,share_espnet_transformer,share_wav2vec2_large,top_member,share_top,entropy,entropy_max,share_sd_mean,switch_rate
achievable_oracle,156155,0,0.9065,0.0214,0.0721,espnet_transducer,0.9065,0.5205,1.5850,0.0009,0.0935
adastt,156155,0,0.4655,0.2754,0.2590,espnet_transducer,0.4655,1.5307,1.5850,0.0179,0.5345
argmin_clf,156155,0,0.8427,0.0153,0.1420,espnet_transducer,0.8427,0.7003,1.5850,0.0031,0.1573
bsm,156155,0,1.0000,0.0000,0.0000,espnet_transducer,1.0000,0.0000,1.5850,0.0000,0.0000
cn_mbr,156155,0,0.3167,0.1093,0.5740,wav2vec2_large,0.5740,1.3341,1.5850,0.0140,0.6833
espnet_transducer,156155,0,1.0000,0.0000,0.0000,espnet_transducer,1.0000,0.0000,1.5850,0.0000,0.0000
espnet_transformer,156155,0,0.0000,1.0000,0.0000,espnet_transformer,1.0000,0.0000,1.5850,0.0000,1.0000
fastt_alternating,156155,0,0.5163,0.2096,0.2741,espnet_transducer,0.5163,1.4767,1.5850,0.0185,0.4837
fastt_boosted,156155,0,0.4003,0.3479,0.2517,espnet_transducer,0.4003,1.5596,1.5850,0.0175,0.5997
kbest_sdt,156155,0,1.0000,0.0000,0.0000,espnet_transducer,1.0000,0.0000,1.5850,0.0000,0.0000


## 6. Significance (Table 6)

Every method against the FASTT anchor (the better of the two FASTT variants on corpus WER) on the same ten folds:
Dietterich's 5×2 cv paired $t$ test and Alpaydın's combined 5×2 cv $F$ test, on corpus WER and on mean utterance
WER.

In [11]:
exp.tests()

Table 6 — every arm against the FASTT anchor, AMI-SDM
on the same 10 folds; positive effect = the anchor is better, in the metric's own units; p < 0.05 in bold. The F-test is the one the paper reports.
  [mean utterance WER — anchor FASTT-Boosted]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
           ESPnet Conformer-RNN-T +0.0306       15.471       0.0000    300.816     0.0000
               ESPnet Transformer +0.0457       15.714       0.0000    237.547     0.0000
              wav2vec2-large-960h +0.1761       47.679       0.0000   2350.760     0.0000
                Best-Single-Model +0.0306       15.471       0.0000    300.816     0.0000
                  Random-Ensemble +0.0850       26.759       0.0000    766.264     0.0000
                 ROVER (weighted) +0.2019       41.609       0.0000   1816.686     0.0000
      ROVER (confidence-weighted) +0.1567      103.354       0.0000  10804.613     0.0000
                           CN-MBR +0.2042       65.294       0.0000   4417.950     0.0000
                     Score-Fusion +0.0826       27.941       0.0000    816.557     0.0000
         Hard-Selector (plain CE) +0.0188       10.072       0.0002    132.057     0.0000
Hard-Selector (cost-sensitive CE) +0.0247        8.267       0.0004     87.017     0.0001
                        QE-Router +0.0423        9.300       0.0002    117.237     0.0000
             Raw XGBoost (AdaSTT) +0.0002       -1.459       0.2045      0.705     0.7025
                          Raw SDT +0.0306       15.471       0.0000    300.816     0.0000
              SelectKBest XGBoost +0.0080        1.748       0.1409     13.878     0.0048
                  SelectKBest SDT +0.0306       15.471       0.0000    300.816     0.0000
                FASTT-Alternating +0.0028        1.657       0.1584      3.757     0.0785
            Oracle (no tolerance) -0.1003      -43.758       0.0000   1858.211     0.0000
        Oracle (1 word tolerance) -0.0513      -36.988       0.0000   1357.565     0.0000
        Oracle (2 word tolerance) -0.0143       -7.301       0.0008     37.824     0.0004
  [corpus WER — anchor FASTT-Boosted]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
           ESPnet Conformer-RNN-T +0.0306       17.722       0.0000    338.551     0.0000
               ESPnet Transformer +0.0781       58.805       0.0000   3454.225     0.0000
              wav2vec2-large-960h +0.0373       25.591       0.0000    715.078     0.0000
                Best-Single-Model +0.0306       17.722       0.0000    338.551     0.0000
                  Random-Ensemble +0.0488       53.352       0.0000   2937.716     0.0000
                 ROVER (weighted) +0.0446        4.318       0.0076     24.937     0.0012
      ROVER (confidence-weighted) +0.0461       20.361       0.0000    441.082     0.0000
                           CN-MBR +0.0459       46.357       0.0000   2219.249     0.0000
                     Score-Fusion +0.0574       39.979       0.0000   1642.878     0.0000
         Hard-Selector (plain CE) +0.0062        3.148       0.0254     15.096     0.0039
Hard-Selector (cost-sensitive CE) +0.0171        3.892       0.0115     17.369     0.0028
                        QE-Router +0.0051        3.861       0.0119     20.767     0.0019
             Raw XGBoost (AdaSTT) +0.0052        2.016       0.0999      8.423     0.0148
                          Raw SDT +0.0306       17.722       0.0000    338.551     0.0000
              SelectKBest XGBoost +0.0122        5.881       0.0020     46.776     0.0003
                  SelectKBest SDT +0.0306       17.722       0.0000    338.551     0.0000
                FASTT-Alternating +0.0018        0.584       0.5844      2.851     0.1295
            Oracle (no tolerance) -0.0637      -59.160       0.0000   3428.327     0.0000
        Oracle (1 word tolerance) -0.0432      -58.314       0.0000   3371.257     0.0000
        Oracle (2 word t

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_ami_sdm.json`;
`manuscript_tables` reads it from the folder `FASTT_RECORDS_DIR` names.

In [12]:
exp.save();

wrote main_ami_sdm.json
